# HCCL 算法简介

上一节认全了算子，本节回答另一半问题：**这些算子是怎么被高效实现的？**

我们从「为什么需要算法」讲起，再逐个看 Ring、Mesh 与 Halving-Doubling 三种典型算法。

## 本节学习大纲

- 为什么需要集合通信算法
- 三种算法总览：都有哪些算法、各自解决什么问题
- Ring 算法（含动画演示）
- Mesh 算法（含动画演示）
- Halving-Doubling 算法（含动画演示）
- 三种算法横向对比
- 课堂小练
- 本节总结
- 课后延伸

## 1. 为什么需要集合通信算法

先把问题摆出来：假设集群里有**上万张卡**，要一起做一次「求平均」。
如果每张卡都自顾自地把数据发给其他所有卡，链路会发生什么？

![图 1](./images/png/why_algorithm.png)

*图 1 · 没有统一流量规划会引发拥塞；集合通信库通过算法统一规划流量*

这就像没有红绿灯的十字路口：车越多，越是互相争抢，最后谁也走不动。
集合通信库的价值就在于此——它通过**通信算法**对流量做统一规划：

- **选择链路**：在多条物理路径中选出合适的路径，避免热点链路被打满；
- **拆分步骤**：把一次大规模通信拆成若干步，让每一步的数据量可控、链路有序交换；
- **适配拓扑**：根据 Star、Tree、Full-mesh、Fat-Tree 等不同物理拓扑选择不同算法。

一句话：**集合通信库实现流量统一规划，在复杂物理拓扑中实现流量有序交换，最大化集群通信性能。**

## 2. 三种算法总览

同一个集合通信算子（比如 AllReduce），底层可以用**不同的算法**去实现。它们的差别只有三条：
**数据怎么走、要走几步、对物理拓扑有什么要求**。

| 算法 | 一句话概括 | 通信步骤 | 每步跟谁交换 | 对拓扑的要求 |
| --- | --- | --- | --- | --- |
| **Ring（环形）** | 所有 Rank 首尾相连成环，**逐跳**接力传递 | **N-1 步** | 固定跟右邻居 | 低——只要能连成环就行 |
| **Mesh（全互联）** | 任意两个 Rank **直连**，所有交换同时进行 | **1 步** | 同时和所有其它 Rank | 极高——必须 Full-mesh |
| **Halving-Doubling** | 每步和**距离翻倍**的邻居交换 | **log₂N 步** | 距离翻倍的伙伴（1、2、4…）| 中——需要支持远距离直连（如 Fat-Tree）|

三者的取舍，其实是同一个矛盾的两面：

- **步数越少 → 延迟越低**，但**对拓扑的要求越高**。Mesh 只要 1 步，可大规模集群做不到两两直连；
- **拓扑要求越低 → 越好落地**，但**步数越多**。Ring 最通用，可规模一大、步数就线性增长；
- **Halving-Doubling 是折中**：步数降到 log₂N，代价是要求拓扑支持「远距离直连」。

> 所以「选哪个算法」没有标准答案——**看你的集群是什么拓扑、要跑多大规模**。
> 下面三节分别把三种算法看一遍，最后再回来做横向对比。

## 3. Ring 算法

Ring（环形）算法把所有 Rank **首尾相连成一个环**。每个 Rank 只和自己的左右邻居说话：
把自己的数据块发给右邻居、同时接收左邻居发来的块。数据像接力一样在环上逐跳流动，
**每完成一跳就换下一块继续传**，共 **N-1 步**后，所有 Rank 都拿齐了全部数据。

下面的动图演示了 4 个 Rank 的完整过程（GIF 动画，VSCode 预览与 GitCode 均可直接播放）。
**只跟一个数据块走一圈**，就能看清 Ring 的全部关键动作：

<img src="./images/gif/ring_anim.gif" alt="ring_anim" width="100%">

*动图 · Ring 算法：一个数据块绕环走一圈，每经过一个 Rank 就归约一次*

看的时候盯住三件事，动图里都画出来了：

- **节点里的「本地块」**——每个 Rank 一开始就持有的那份数据，不用谁来送；
- **块上的标签会变**：`P1` → `P1+P2` → `P1+P2+P3` → `P1+P2+P3+P0`。
  每经过一个 Rank，块就和当地的本地块**做一次归约（按元素相加）**，所以内容越滚越多；
- **底部「归约演算台」**，给出这一步的算式，例如 `60 + 40 = 100`：
  左边是**传过来的块**、中间是**当地本地块**、右边是**算出的新本地块**（本地块会被它覆盖）。

### 3.1 Ring 算法的两个关键参数

- **适用拓扑**：Star、Tree 等规模。Ring 对拓扑要求不高，任何能连成环的结构都能用；
- **通信步骤**：**N-1 步**（N 为通信域内的 Rank 数）。

> **为什么是 N-1 步？** 每个 Rank 已经有 1 份**本地块**（自己的那份，不用再传给自己），
> 还差另外 **N-1 份**要从邻居那里收齐；每收一份需要 1 跳，所以总共 N-1 跳。
> 走完以后，每个 Rank 的本地块都变成了"所有人数据的总和"，于是各卡结果相同。
>
> **优点**：每一步每张卡都只与一个邻居通信，链路复用充分、实现简单、扩展性好；
> **代价**：步数随 N 线性增长，大规模下总延迟较高——这正是 Mesh 与 Halving-Doubling 要解决的问题。

## 4. Mesh 算法

Ring 用「逐跳」换来了「低拓扑要求」，但步数随规模线性增长。如果物理拓扑足够好——
**任意两个 Rank 之间都有直连链路**——就可以让所有人**同时**互相交换，**1 步**解决问题。

<img src="./images/gif/mesh_anim.gif" alt="mesh_anim" width="100%">

*动图 · Mesh 算法：所有 Rank 同时互相交换，1 步完成*

**Mesh 的四个特点**：

| 特点 | 说明 |
| --- | --- |
| 两两直连 | 任意两个 Rank 之间都有独立链路，不用经过第三方转发 |
| 全并发 | 所有交换**同时**进行，没有先后等待 |
| **只要 1 步** | 没有逐跳转发，所以延迟最低 |
| 拓扑要求极高 | 必须是 Full-mesh，大规模集群很难做到两两直连 |

> **和 Ring 的关系**：结果完全一样（每个 Rank 的本地块都变成全部数据的累加），
> 区别只在**用了 1 步而不是 N-1 步**——省下来的时间，是用「拓扑成本」换的。

## 5. Halving-Doubling 算法

Mesh 快，但要求太高；Ring 通用，但步数太多。**Halving-Doubling 走的是中间路线**：
每一步都和一个**距离翻倍**的伙伴交换——第 1 步距离 1，第 2 步距离 2，第 3 步距离 4……

因为距离是翻倍增长的，所以只要 **log₂N 步**就能让所有人拿齐数据。

<img src="./images/gif/hd_anim.gif" alt="hd_anim" width="100%">

*动图 · Halving-Doubling 算法：步长按 2^k 翻倍，log₂N 步完成（图中 4 个 Rank，2 步）*

**怎么读这张动图**：

- **第 1 步（距离 1）**：相邻的两两配对——(0↔1)、(2↔3) 同时交换，交换完各拿到「两份之和」；
- **第 2 步（距离 2）**：改成隔一个配对——(0↔2)、(1↔3) 同时交换，各拿到「四份之和」；
- 每步的**距离翻倍**（1、2、4…），所以 N=4 只要 **2 步**（= log₂4），
  N=8 只要 3 步——而 Ring 要 7 步。

| 对比项 | Ring | Halving-Doubling |
| --- | --- | --- |
| 通信步骤 | N-1 步 | **log₂N 步** |
| 每步跟谁换 | 固定跟右邻居 | 距离翻倍（1、2、4…）|
| 对拓扑的要求 | 低 | 中——需要支持远距离直连（如 Fat-Tree）|

> **代价**：它要求拓扑支持「远距离直连」。如果是那种只能相邻通信的拓扑，
> 这一步跨两个 Rank 的交换就得绕路，优势也就没了——**算法必须和拓扑匹配**。

## 6. 三种算法横向对比

把前面三节的内容收成一张表：

| 算法 | 适用拓扑 | 通信步骤 | 每步跟谁交换 | 特点 |
| --- | --- | --- | --- | --- |
| **Ring** | Star、Tree 等 | **N-1 步** | 固定跟右邻居，逐跳接力 | 链路复用充分、实现简单、扩展性好；步数随规模线性增长 |
| **Mesh（Full-Mesh）** | Full-mesh | **仅 1 步** | 同时和**所有**其它 Rank | 延迟最低；但拓扑要求极高，大规模难满足 |
| **Halving-Doubling** | Fat-Tree 等 | **log₂N 步** | 距离翻倍的伙伴（1、2、4…）| 步数与拓扑要求之间取折中，大规模 Fat-Tree 常用 |

> **选型直觉**：拓扑越好、规模越大，越倾向于用**步数更少**的算法。
> 小规模或拓扑受限时，Ring 简单可靠；大规模 Fat-Tree 集群则常用 Halving-Doubling。
> 实际实现中，HCCL 会根据**拓扑 + 数据量**自动选择合适的算法。

### 6.1 这些算法不只服务 AllReduce

前面几节都用 AllReduce 举例，但**同一套算法骨架可以服务多个算子**——
区别只在**每一步搬的是什么数据、搬完要不要做归约**。

| 算法 | 也能服务这些算子 | 每步搬什么 / 做什么 |
| --- | --- | --- |
| **Ring** | AllReduce、ReduceScatter、AllGather、AlltoAll | 都是「沿环传一块」；**传之前要不要先归约、收完要不要再广播**，决定了它是哪个算子 |
| **Mesh** | AllGather、AllReduce、AlltoAll | 都是「一次性发给所有人」；收完是**拼起来**还是**归约**，决定了它是哪个算子 |
| **Halving-Doubling** | AllReduce、ReduceScatter、AllGather | 它本身就是「**归约 + 广播**」的组合：前一半做 ReduceScatter、后一半做 AllGather |

最直观的一个例子：

> **AllReduce = ReduceScatter + AllGather**

放在 Ring 上看，这就是**同一套环走两遍**：前半程一边传一边归约（ReduceScatter），
后半程只传不归约（AllGather）。所以「Ring 能实现 AllReduce」和「Ring 能实现
ReduceScatter / AllGather」，说的其实是同一件事。

不过还有两点要注意：

1. **不是所有算子都适合所有算法。** All2All 的本质是「每个人都给其他每个人各发一份**不同的**
   数据」，用 Ring 逐跳传要绕很多圈，**用 Mesh 一次直连发完**才自然。
   所以实际选型是**算子 × 拓扑一起看**的，不是「挑个最快的算法」。
2. **这三种只是最常用的入门算法。** 除此之外还有：

| 其他算法 | 大致用来做什么 |
| --- | --- |
| **Tree（树形 / 递归倍增）** | 适合 Broadcast、Reduce 这类「一对多 / 多对一」：数据沿树逐层展开或收拢 |
| **Butterfly（蝶形）** | 每一轮伙伴两两配对、逐轮换伙伴，和 Halving-Doubling 同源，适合 AllGather 类 |
| **网内归约（交换机侧计算）** | 把归约放到交换机里做，不占计算卡，适合超大规模集群 |

> 算法越多样，越说明同一件事：**没有万能算法，只有和「拓扑 + 数据量 + 算子」都匹配的算法。**

## 7. 课堂小练

请根据本节学习内容完成以下题目进行自测。

**1、（判断题）** 只要物理拓扑是 Full-mesh，Mesh 算法就只需 1 步即可完成一次集合通信。

---

**2、（填空题）** Ring 算法的通信步骤是 ______ 步；Halving-Doubling 算法的通信步骤是 ______ 步，
其访问步长按照 ______ 增长。

---

**3、（简答题）** 为什么需要集合通信算法？请从「流量规划」与「拓扑适配」两个角度简要说明。

---

**4、（判断题）** Halving-Doubling 算法每一步都和「固定距离」的邻居交换。

---

**5、（单选题）** Ring 算法里，一个数据块经过一个 Rank 之后，块里的内容会发生什么？

- **A)** 保持不变，只是换了个位置
- **B)** 被该 Rank 的本地块替换掉
- **C)** 与该 Rank 的本地块做归约（按元素相加），内容越滚越多
- **D)** 被拆成两半，各留一份

---

**6、（单选题）** 下列三种算法中，对物理拓扑的要求**最高**的是？

- **A)** Ring
- **B)** Mesh（Full-mesh）
- **C)** Halving-Doubling
- **D)** 三者一样

---

**7、（判断题）** 选哪种算法，只要看算法本身快不快就行，和集群的物理拓扑无关。

---

**8、（简答题）** 一个 8 卡的集群里，Ring 和 Halving-Doubling 各需要多少步？
这个对比说明了什么？

**执行以下代码获取答案。**

In [ ]:
# 读取本节课堂小练答案（用 Python 读文件：跨平台，并保证中文不乱码）
import pathlib
_p = pathlib.Path('./answer/02.03_answer.txt')
if _p.exists():
    print(_p.read_text(encoding='utf-8'))
else:
    print(f'未找到答案文件：{_p}')
    print('（请确认 answer/ 目录与本 notebook 在同一层）')

## 8. 本节总结

### 为什么需要算法

- **流量规划**：上万张卡同时发数据会互相争抢、链路拥塞；算法把一次通信拆成若干步，让每步可控；
- **拓扑适配**：Star / Tree / Full-mesh / Fat-Tree 各不相同，没有一种走法在所有拓扑上都高效。

### 三种算法

| 算法 | 通信步骤 | 每步跟谁交换 | 对拓扑的要求 |
| --- | --- | --- | --- |
| **Ring** | **N-1 步** | 固定跟右邻居，逐跳接力 | 低 |
| **Mesh** | **仅 1 步** | 同时和所有其它 Rank | 极高（Full-mesh）|
| **Halving-Doubling** | **log₂N 步** | 距离翻倍的伙伴（1、2、4…）| 中（如 Fat-Tree）|

### 一条主线

- 数据块每经过一个 Rank，就和当地的**本地块做一次归约**，内容越滚越多；
- **步数越少 → 延迟越低，但拓扑要求越高**；三者就是在这一对矛盾上取不同位置；
- 实际实现中，HCCL 会根据**拓扑 + 数据量**自动选算法——**算法必须和拓扑匹配**。

### 算法与算子的关系

- **同一套算法骨架能服务多个算子**，区别只在每步搬什么、搬完要不要归约；
- 最直观的例子：**AllReduce = ReduceScatter + AllGather**——在 Ring 上就是同一套环走两遍；
- 但**算子 × 拓扑要一起看**：All2All 用 Mesh 直连最自然，用 Ring 逐跳就很吃亏；
- 除这三种，还有 **Tree、Butterfly、网内归约** 等——**没有万能算法，只有匹配的算法**。

## 9. 课后延伸

| 资源 | 地址 |
| --- | --- |
| HCCL 仓（集合通信库 · 算法 / 算子） | https://gitcode.com/cann/hccl |
| HCOMM 仓（通信基础库） | https://gitcode.com/cann/hcomm |
| HCCL SIG 组（社区需求 / 讨论 / 活动） | https://gitcode.com/cann/community |
| CANN 学习中心 | https://gitcode.com/cann/cann-learning-hub |

<img src="./images/qrcode_cann_community.jpg" alt="CANN 社区" width="18%">

### 上一章回顾

**第 1 章 · HCCL 简介与集合通信基础**：HCCL 是什么、通信域组织通信成员、通信算子承载通信算法，
以及单卡瓶颈、四类并行方式与 MoE 专家并行的入门认知。

### 下一章预告

**第 3 章 · HCCL 软件架构与编程模型**：认识了算子与算法之后，下一步要回答——
这些能力在软件上是**如何分层、如何被调用**的？HCCL 的软件架构与编程模型将从这里展开。